In [0]:
import dlt
from pyspark.sql.functions import col, to_date

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-5317772545711028>, line 1
----> 1 import dlt
      2 from pyspark.sql.functions import col, to_date
      4 # -------------------------------------------------------------------
      5 # Target streaming table (merge target)
      6 # -------------------------------------------------------------------

File /databricks/python_shell/dbruntime/autoreload/discoverability/hook.py:72, in AutoreloadDiscoverabilityHook.pre_run_cell.<locals>.patched_import(name, *args, **kwargs)
     66 if not self._should_hint and (
     67     (module := sys.modules.get(absolute_name)) is not None and
     68     (fname := get_allowed_file_name_or_none(module)) is not None and
     69     (mtime := os.stat(fname).st_mtime) > self.last_mtime_by_modname.get(
     70         absolute_name, float("inf")) and not self._should_hint):
     71     self

In [0]:
@dlt.view(
    name="electroniz_bronze_inventory_view",
    comment="Streaming view with transformed inventory data"
)
def electroniz_bronze_inventory_view():
    df = spark.readStream.table("electroniz_catalog.electroniz_bronze_schema.electroniz_bronze_inventory")
    return df.select(
        to_date(col("inventory_date"), "dd/MM/yyyy").alias("inventory_date"),
        col("product").alias("product_name"),
        col("inventory"),
        col("updated_at")
    )

In [0]:
dlt.create_streaming_table(
    name="electroniz_silver_inventory",
    comment="Silver inventory table",
    table_properties={"quality": "silver"}
)

dlt.apply_changes(
    target="electroniz_silver_inventory",
    source="electroniz_bronze_inventory_view",
    keys=["product_name"],
    sequence_by=col("updated_at"),
    stored_as_scd_type=1
)